# NFL Player Contact Detection: EDA and Tracking Context

This notebook establishes the evidence base for player contact detection. It profiles the contact labels, tracking data, helmet boxes, and video metadata; then it builds a first tracking-distance baseline that later modeling notebooks can improve.


## 1. Setup and Configuration

Imports, plotting defaults, path resolution, and fast-run flags live together. The resolver checks Kaggle's `competitions` mount first, then dataset-style fallbacks, and finally scans the visible input directories for the required CSV files.


In [ ]:
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from IPython.display import display
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

RANDOM_STATE = 42
TARGET = "contact"
ID_COL = "contact_id"
RUN_FAST = True
FAST_SAMPLE_PLAYS = 30
FAST_SAMPLE_ROWS = 200_000
DISTANCE_THRESHOLDS = np.round(np.arange(0.2, 3.05, 0.1), 2)
VIDEO_FPS = 59.94
TRACKING_HZ = 10
SNAP_OFFSET_SECONDS = 5
SNAP_FRAME = int(round(VIDEO_FPS * SNAP_OFFSET_SECONDS))

REQUIRED_DATA_FILES = [
    "train_labels.csv",
    "sample_submission.csv",
    "train_player_tracking.csv",
    "test_player_tracking.csv",
    "train_baseline_helmets.csv",
    "test_baseline_helmets.csv",
    "train_video_metadata.csv",
    "test_video_metadata.csv",
]

DATA_DIR = Path("/kaggle/input/competitions/nfl-player-contact-detection")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

missing_files = [
    filename for filename in REQUIRED_DATA_FILES
    if not (DATA_DIR / filename).exists()
]
if missing_files:
    raise FileNotFoundError(
        f"Missing required files in {DATA_DIR}: {missing_files}"
    )

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")


## 2. Helper Functions

The helpers keep contact ID parsing, memory reduction, plotting, distance scoring, and contact-run summaries consistent across the EDA and baseline notebooks.


In [ ]:
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast dataframe columns to reduce notebook memory usage."""
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5 and col != ID_COL:
                out[col] = out[col].astype("category")
    return out


def parse_contact_id(df: pd.DataFrame) -> pd.DataFrame:
    """Split Kaggle contact_id into play, step, player1, and player2 fields."""
    out = df.copy()
    parts = out[ID_COL].astype(str).str.split("_", expand=True)
    if parts.shape[1] != 5:
        raise ValueError(f"Expected 5 contact_id parts, found {parts.shape[1]}")
    out["game_play"] = parts[0] + "_" + parts[1]
    out["step"] = parts[2].astype("int16")
    out["nfl_player_id_1"] = parts[3].astype(str)
    out["nfl_player_id_2"] = parts[4].astype(str)
    out["contact_type"] = np.where(
        out["nfl_player_id_2"].eq("G"), "ground", "player_player"
    )
    out["pair_key"] = (
        out["game_play"].astype(str) + "_"
        + out["nfl_player_id_1"].astype(str) + "_"
        + out["nfl_player_id_2"].astype(str)
    )
    return out


def dataframe_overview(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Summarize column types, missingness, unique values, and memory."""
    return pd.DataFrame({
        "dataset": name,
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=False),
        "memory_mb": df.memory_usage(deep=True).div(1024 ** 2),
    }).sort_values(["missing_pct", "unique"], ascending=[False, False])


def maybe_sample_plays(df: pd.DataFrame, n_plays: int) -> pd.DataFrame:
    """Return all rows for a reproducible sample of game_play values."""
    plays = pd.Series(df["game_play"].unique()).sample(
        min(n_plays, df["game_play"].nunique()),
        random_state=RANDOM_STATE,
    )
    return df[df["game_play"].isin(plays)].copy()


def maybe_sample_rows(df: pd.DataFrame, n_rows: int) -> pd.DataFrame:
    """Sample rows reproducibly when plotting very large tables."""
    if len(df) <= n_rows:
        return df.copy()
    return df.sample(n_rows, random_state=RANDOM_STATE).copy()


def create_football_field(
    linenumbers: bool = True,
    endzones: bool = True,
    figsize: tuple[int, int] = (12, 6),
    line_color: str = "black",
    field_color: str = "white",
    endzone_color=None,
    ax=None,
):
    """Draw a football field for tracking-data visualization."""
    if endzone_color is None:
        endzone_color = field_color
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    rect = patches.Rectangle(
        (0, 0), 120, 53.3, linewidth=0.1, edgecolor=line_color,
        facecolor=field_color, zorder=0,
    )
    ax.add_patch(rect)
    for x in range(10, 120, 10):
        ax.plot([x, x], [0, 53.3], color=line_color, linewidth=0.8)
    if endzones:
        ax.add_patch(patches.Rectangle(
            (0, 0), 10, 53.3, linewidth=0.1, edgecolor=line_color,
            facecolor=endzone_color, alpha=0.5, zorder=0,
        ))
        ax.add_patch(patches.Rectangle(
            (110, 0), 10, 53.3, linewidth=0.1, edgecolor=line_color,
            facecolor=endzone_color, alpha=0.5, zorder=0,
        ))

    if linenumbers:
        for x in range(20, 110, 10):
            number = x if x <= 50 else 120 - x
            label = str(number - 10)
            ax.text(x, 5, label, ha="center", fontsize=16, color=line_color)
            ax.text(
                x, 48.3, label, ha="center", fontsize=16,
                color=line_color, rotation=180,
            )

    hash_range = range(11, 110) if endzones else range(1, 120)
    for x in hash_range:
        ax.plot([x, x], [0.4, 0.7], color=line_color, linewidth=0.6)
        ax.plot([x, x], [52.6, 52.9], color=line_color, linewidth=0.6)
        ax.plot([x, x], [22.91, 23.57], color=line_color, linewidth=0.6)
        ax.plot([x, x], [29.73, 30.39], color=line_color, linewidth=0.6)

    ax.set_xlim(-5, 125)
    ax.set_ylim(-5, 58.3)
    ax.axis("off")
    return ax


def compute_pair_distances(
    contacts: pd.DataFrame,
    tracking: pd.DataFrame,
) -> pd.DataFrame:
    """Attach player coordinates and compute player-player distance in yards."""
    pairs = contacts[contacts["nfl_player_id_2"].ne("G")].copy()
    track_cols = [
        "game_play", "step", "nfl_player_id", "x_position", "y_position"
    ]
    track = tracking[track_cols].copy()
    for col in ["nfl_player_id", "nfl_player_id_1", "nfl_player_id_2"]:
        if col in track.columns:
            track[col] = track[col].astype(str)
        if col in pairs.columns:
            pairs[col] = pairs[col].astype(str)

    out = pairs.merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_1"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).rename(columns={
        "x_position": "x_position_1",
        "y_position": "y_position_1",
    }).drop(columns=["nfl_player_id"])

    out = out.merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_2"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).rename(columns={
        "x_position": "x_position_2",
        "y_position": "y_position_2",
    }).drop(columns=["nfl_player_id"])

    out["distance"] = np.sqrt(
        np.square(out["x_position_1"] - out["x_position_2"])
        + np.square(out["y_position_1"] - out["y_position_2"])
    )
    return out


def score_distance_thresholds(
    df: pd.DataFrame,
    thresholds: np.ndarray,
) -> pd.DataFrame:
    """Score distance thresholds while predicting ground rows as no-contact."""
    rows = []
    y_true = df[TARGET].astype(int).to_numpy()
    contact_type = df["contact_type"].to_numpy()
    distances = df["distance"].to_numpy()
    is_pair = contact_type == "player_player"

    for threshold in thresholds:
        pred = np.zeros(len(df), dtype=np.int8)
        pred[is_pair] = np.nan_to_num(distances[is_pair], nan=np.inf) <= threshold
        rows.append({
            "threshold": threshold,
            "mcc": matthews_corrcoef(y_true, pred),
            "positive_rate": pred.mean(),
        })
    return pd.DataFrame(rows).sort_values("mcc", ascending=False)


def summarize_contact_runs(labels_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize contiguous positive contact runs by play/pair/type."""
    positives = labels_df.loc[
        labels_df[TARGET].eq(1),
        ["game_play", "pair_key", "contact_type", "step"],
    ].sort_values(["pair_key", "step"])
    if positives.empty:
        return positives.assign(run_length_steps=[])

    new_run = (
        positives["pair_key"].ne(positives["pair_key"].shift())
        | positives["step"].sub(positives["step"].shift()).ne(1)
    )
    positives = positives.assign(run_id=new_run.cumsum())
    return (
        positives.groupby(["game_play", "pair_key", "contact_type", "run_id"], observed=True)
        .agg(
            start_step=("step", "min"),
            end_step=("step", "max"),
            run_length_steps=("step", "size"),
        )
        .reset_index()
        .assign(duration_seconds=lambda df: df["run_length_steps"] / TRACKING_HZ)
    )


## 3. Load Data

Read the core competition tables. Both train and test metadata are loaded because the scoring notebook will receive test tracking, helmet, metadata, and submission rows at inference time.


In [ ]:
labels = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_labels.csv",
    parse_dates=["datetime"],
))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_player_tracking.csv",
    parse_dates=["datetime"],
))
test_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "test_player_tracking.csv",
    parse_dates=["datetime"],
))
train_helmets = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_baseline_helmets.csv",
))
test_helmets = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "test_baseline_helmets.csv",
))
train_video_metadata = pd.read_csv(
    DATA_DIR / "train_video_metadata.csv",
    parse_dates=["start_time", "end_time", "snap_time"],
)
test_video_metadata = pd.read_csv(
    DATA_DIR / "test_video_metadata.csv",
    parse_dates=["start_time", "end_time", "snap_time"],
)

labels = parse_contact_id(labels)
sample_submission = parse_contact_id(sample_submission)

shape_summary = pd.DataFrame({
    "dataset": [
        "labels", "sample_submission", "train_tracking", "test_tracking",
        "train_helmets", "test_helmets", "train_video_metadata",
        "test_video_metadata",
    ],
    "rows": [
        len(labels), len(sample_submission), len(train_tracking), len(test_tracking),
        len(train_helmets), len(test_helmets), len(train_video_metadata),
        len(test_video_metadata),
    ],
    "columns": [
        labels.shape[1], sample_submission.shape[1], train_tracking.shape[1],
        test_tracking.shape[1], train_helmets.shape[1], test_helmets.shape[1],
        train_video_metadata.shape[1], test_video_metadata.shape[1],
    ],
})
display(shape_summary)
labels.head()


### Current Run Takeaway

The latest Kaggle run loaded `4,721,618` label rows, `1,353,053` train tracking rows, `3,783,616` train helmet rows, and `480` train video metadata rows. The overall contact rate was about `1.37%`, so MCC and recall-oriented diagnostics matter much more than accuracy.


## 4. Data Quality and Schema Checks

Before interpreting signals, check missingness, duplicates, ID parsing, file coverage, and train/test schema alignment.


In [ ]:
quality_checks = pd.DataFrame({
    "check": [
        "duplicated label rows",
        "duplicated contact_id",
        "label game_play count",
        "train tracking game_play count",
        "test sample game_play count",
        "sample duplicated contact_id",
        "train/test tracking shared columns",
        "train/test helmet shared columns",
    ],
    "value": [
        int(labels.duplicated().sum()),
        int(labels[ID_COL].duplicated().sum()),
        int(labels["game_play"].nunique()),
        int(train_tracking["game_play"].nunique()),
        int(sample_submission["game_play"].nunique()),
        int(sample_submission[ID_COL].duplicated().sum()),
        len(set(train_tracking.columns).intersection(test_tracking.columns)),
        len(set(train_helmets.columns).intersection(test_helmets.columns)),
    ],
})

display(pd.concat([
    dataframe_overview(labels, "labels"),
    dataframe_overview(train_tracking, "train_tracking"),
    dataframe_overview(train_helmets, "train_helmets"),
]))
display(quality_checks)

schema_checks = {
    "tracking_train_only": sorted(set(train_tracking.columns) - set(test_tracking.columns)),
    "tracking_test_only": sorted(set(test_tracking.columns) - set(train_tracking.columns)),
    "helmet_train_only": sorted(set(train_helmets.columns) - set(test_helmets.columns)),
    "helmet_test_only": sorted(set(test_helmets.columns) - set(train_helmets.columns)),
    "metadata_train_only": sorted(set(train_video_metadata.columns) - set(test_video_metadata.columns)),
    "metadata_test_only": sorted(set(test_video_metadata.columns) - set(train_video_metadata.columns)),
}
display(pd.Series(schema_checks, name="column_differences"))


## 5. Contact Label Balance

The task combines two related but distinct problems: player-player contact and player-ground contact. This section measures the imbalance and play-level variation that validation needs to preserve.


In [ ]:
label_summary = (
    labels.groupby("contact_type", observed=True)[TARGET]
    .agg(["count", "mean", "sum"])
    .rename(columns={"mean": "contact_rate", "sum": "positive_count"})
)
display(label_summary)
print(f"Overall contact rate: {labels[TARGET].mean():.5f}")

play_summary = (
    labels.groupby("game_play", observed=True)
    .agg(
        rows=(TARGET, "size"),
        positive_contacts=(TARGET, "sum"),
        contact_rate=(TARGET, "mean"),
        max_step=("step", "max"),
        players=("nfl_player_id_1", "nunique"),
        pairs=("pair_key", "nunique"),
    )
    .assign(play_seconds=lambda df: (df["max_step"] + 1) / TRACKING_HZ)
    .sort_values("contact_rate", ascending=False)
)
display(play_summary.head(12))
display(play_summary.describe().T)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=labels, x=TARGET, color=sns.color_palette("viridis", 8)[4], ax=axes[0])
axes[0].set_title("Overall label balance")
sns.barplot(
    data=label_summary.reset_index(),
    x="contact_type",
    y="contact_rate",
    palette="viridis",
    ax=axes[1],
)
axes[1].set_title("Contact rate by type")
axes[1].set_xlabel("")
sns.histplot(play_summary["contact_rate"], bins=40, color=sns.color_palette("viridis", 8)[4], ax=axes[2])
axes[2].set_title("Play-level contact-rate variation")
plt.tight_layout()
plt.show()


### Label Takeaway

The positive class is very rare overall, but contact is clustered by play, pair, and time. Validation should therefore remain grouped by `game_play`; random row splits would overstate performance by sharing nearly identical moments across train and validation.


## 6. Temporal Contact Dynamics

Labels are provided at 10 Hz and may be noisy within roughly one timestep. Run-length and step-level views reveal how long contacts persist and where in plays contacts concentrate.


In [ ]:
step_summary = (
    labels.groupby(["contact_type", "step"], observed=True)[TARGET]
    .agg(["count", "sum", "mean"])
    .rename(columns={"sum": "positive_count", "mean": "contact_rate"})
    .reset_index()
)
contact_runs = summarize_contact_runs(labels)
run_summary = (
    contact_runs.groupby("contact_type", observed=True)["duration_seconds"]
    .agg(["count", "mean", "median", "max"])
    .rename(columns={"count": "positive_runs"})
)

display(run_summary)
display(contact_runs.sort_values("duration_seconds", ascending=False).head(12))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.lineplot(
    data=step_summary,
    x="step",
    y="contact_rate",
    hue="contact_type",
    palette="viridis",
    ax=axes[0],
)
axes[0].set_title("Contact rate by step")
axes[0].set_ylabel("contact rate")
plot_runs = contact_runs.query("duration_seconds <= 5")
sns.histplot(
    data=plot_runs,
    x="duration_seconds",
    hue="contact_type",
    bins=40,
    palette="viridis",
    multiple="layer",
    ax=axes[1],
)
axes[1].set_title("Positive contact run duration")
plt.tight_layout()
plt.show()


## 7. Tracking Context and Motion Slices

Tracking features explain body location and movement at 10 Hz. Ground contact likely needs motion and posture proxies, while player-player contact starts with pair distance and relative movement.


In [ ]:
tracking_numeric = train_tracking.select_dtypes(include=[np.number]).columns.tolist()
display(train_tracking[tracking_numeric].describe().T)

plot_cols = [
    col for col in [
        "x_position", "y_position", "speed", "distance",
        "acceleration", "sa", "direction", "orientation",
    ]
    if col in train_tracking.columns
]
tracking_plot = maybe_sample_rows(train_tracking, FAST_SAMPLE_ROWS)
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
axes = np.ravel(axes)
for ax, col in zip(axes, plot_cols):
    sns.histplot(
        tracking_plot[col].dropna(),
        bins=60,
        color=sns.color_palette("viridis", 8)[4],
        ax=ax,
    )
    ax.set_title(col)
for ax in axes[len(plot_cols):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

motion_cols = [
    col for col in ["speed", "distance", "acceleration", "sa"]
    if col in train_tracking.columns
]
ground_labels = labels.loc[
    labels["contact_type"].eq("ground"),
    [ID_COL, "game_play", "step", "nfl_player_id_1", TARGET],
].copy()
ground_sample = maybe_sample_rows(ground_labels, FAST_SAMPLE_ROWS)
ground_motion = ground_sample.merge(
    train_tracking[["game_play", "step", "nfl_player_id"] + motion_cols].assign(
        nfl_player_id=lambda df: df["nfl_player_id"].astype(str)
    ),
    left_on=["game_play", "step", "nfl_player_id_1"],
    right_on=["game_play", "step", "nfl_player_id"],
    how="left",
)
if motion_cols:
    display(ground_motion.groupby(TARGET)[motion_cols].agg(["mean", "median", "std"]))


## 8. Player-Player Distance Deep Dive

The starter baseline uses tracking distance. Here we inspect the distance distribution and contact rate by distance bin before converting it into a hard-threshold MCC baseline.


In [ ]:
distance_labels = labels[[
    ID_COL, TARGET, "game_play", "step", "nfl_player_id_1",
    "nfl_player_id_2", "contact_type", "pair_key",
]].copy()
if RUN_FAST:
    distance_labels = maybe_sample_plays(distance_labels, FAST_SAMPLE_PLAYS)

distance_pairs = compute_pair_distances(distance_labels, train_tracking)
distance_scoring = distance_labels.merge(
    distance_pairs[[ID_COL, "distance"]],
    on=ID_COL,
    how="left",
)

pair_distance = distance_scoring.query("contact_type == 'player_player'").copy()
pair_distance["distance_bin"] = pd.cut(
    pair_distance["distance"],
    bins=[0, 0.5, 1, 1.5, 2, 3, 5, 10, 120],
    include_lowest=True,
)
distance_bin_summary = (
    pair_distance.groupby("distance_bin", observed=True)[TARGET]
    .agg(["count", "sum", "mean"])
    .rename(columns={"sum": "positive_count", "mean": "contact_rate"})
)
display(distance_bin_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(
    data=pair_distance.query("distance <= 10"),
    x="distance",
    hue=TARGET,
    bins=60,
    palette="viridis",
    stat="density",
    common_norm=False,
    ax=axes[0],
)
axes[0].set_title("Player-player distance distribution <= 10 yards")
sns.barplot(
    data=distance_bin_summary.reset_index(),
    x="distance_bin",
    y="contact_rate",
    palette="viridis",
    ax=axes[1],
)
axes[1].set_title("Contact rate by distance bin")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 9. Helmet and Video Metadata Context

Baseline helmet boxes are imperfect but valuable for connecting video evidence to tracked players. Video metadata has timing fields, while the helmet files carry the actual video filenames.


In [ ]:
helmet_summary = pd.DataFrame({
    "dataset": ["train_helmets", "test_helmets"],
    "rows": [len(train_helmets), len(test_helmets)],
    "videos": [train_helmets["video"].nunique(), test_helmets["video"].nunique()],
    "game_plays": [train_helmets["game_play"].nunique(), test_helmets["game_play"].nunique()],
    "players": [train_helmets["nfl_player_id"].nunique(), test_helmets["nfl_player_id"].nunique()],
})
display(helmet_summary)

video_summary = pd.concat([
    train_video_metadata.assign(dataset="train"),
    test_video_metadata.assign(dataset="test"),
], ignore_index=True)
video_summary["duration_seconds"] = (
    video_summary["end_time"] - video_summary["start_time"]
).dt.total_seconds()
video_summary["snap_offset_seconds"] = (
    video_summary["snap_time"] - video_summary["start_time"]
).dt.total_seconds()
video_summary["estimated_frames"] = (
    video_summary["duration_seconds"] * VIDEO_FPS
).round().astype("int")
display(video_summary.head())
display(video_summary.groupby(["dataset", "view"], observed=True).agg(
    rows=("game_play", "size"),
    game_plays=("game_play", "nunique"),
    median_duration_seconds=("duration_seconds", "median"),
    median_snap_offset_seconds=("snap_offset_seconds", "median"),
    median_estimated_frames=("estimated_frames", "median"),
))

helmet_frame_summary = (
    train_helmets.groupby(["view", "video"], observed=True)
    .agg(
        frames=("frame", "nunique"),
        boxes=("frame", "size"),
        players=("nfl_player_id", "nunique"),
    )
    .assign(boxes_per_frame=lambda df: df["boxes"] / df["frames"])
    .reset_index()
)
display(helmet_frame_summary.groupby("view", observed=True).agg(
    videos=("video", "nunique"),
    median_frames=("frames", "median"),
    median_players=("players", "median"),
    median_boxes_per_frame=("boxes_per_frame", "median"),
))

box_cols = [c for c in ["left", "width", "top", "height"] if c in train_helmets]
if box_cols:
    display(train_helmets[box_cols].describe().T)


## 10. Example Field View

Plot one play and step on a football field to verify coordinate orientation and team separation.


In [ ]:
example_game_play = labels["game_play"].iloc[0]
example_step = int(labels.loc[labels["game_play"].eq(example_game_play), "step"].median())
example_tracking = train_tracking.query(
    "game_play == @example_game_play and step == @example_step"
).copy()

fig, ax = plt.subplots(figsize=(12, 6))
create_football_field(ax=ax)
if len(example_tracking):
    sns.scatterplot(
        data=example_tracking,
        x="x_position",
        y="y_position",
        hue="team",
        palette="viridis",
        s=80,
        ax=ax,
    )
    ax.set_title(f"{example_game_play}, step {example_step}")
plt.show()


## 11. Distance Baseline Sanity Check

Tune player-player distance thresholds with a play-grouped validation split. Ground rows are predicted as no-contact in this first baseline, making the limitation explicit.


In [ ]:
work_labels = labels[[
    ID_COL, TARGET, "game_play", "step", "nfl_player_id_1",
    "nfl_player_id_2", "contact_type", "pair_key",
]].copy()
if RUN_FAST:
    work_labels = maybe_sample_plays(work_labels, FAST_SAMPLE_PLAYS)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE,
)
_, valid_idx = next(splitter.split(work_labels, groups=work_labels["game_play"]))
valid_labels = work_labels.iloc[valid_idx].copy()

valid_distances = compute_pair_distances(valid_labels, train_tracking)
valid_scoring = valid_labels.merge(
    valid_distances[[ID_COL, "distance"]],
    on=ID_COL,
    how="left",
)
threshold_scores = score_distance_thresholds(
    valid_scoring,
    DISTANCE_THRESHOLDS,
)
best_threshold = float(threshold_scores.iloc[0]["threshold"])
best_mcc = float(threshold_scores.iloc[0]["mcc"])

print(f"Validation rows: {len(valid_scoring):,}")
print(f"Validation plays: {valid_scoring['game_play'].nunique():,}")
print(f"Best threshold: {best_threshold:.2f} yards")
print(f"Best MCC: {best_mcc:.5f}")
display(threshold_scores.head(10))

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(
    data=threshold_scores.sort_values("threshold"),
    x="threshold",
    y="mcc",
    marker="o",
    ax=ax,
)
ax.axvline(best_threshold, color="black", linestyle="--", linewidth=1)
ax.set_title("Distance threshold validation MCC")
plt.show()

gc.collect()


### EDA Conclusions and Modeling Implications

The first EDA pass points to a two-branch solution. Player-player contact should start with distance, relative motion, and temporal smoothing. Player-ground contact needs a dedicated motion/video branch because player-player distance cannot detect it. Helmet boxes and synchronized Sideline/Endzone videos should be added after the tracking-only validation harness is stable.
